In [ ]:
prefix_path = '../..'

import sys
sys.path.append(prefix_path)

import logging
import os
import json
import torch
from torch.utils.data import DataLoader

from detection_baselines.detection_attention import (
    load_attention_dataset, attention_collate_fn,
    AttentionThresholdDetector, train_attention_detector, eval_attention_detector
)
from egh_vlm.utils import load_phd_dataset, split_stratified

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3_vl_2b.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/attention_phd_qwen3_layer24.pt'
OUTPUT_PATH = f'{prefix_path}/data/phd/evaluations/evaluation_attention_phd_qwen3_layer24.json'
LOGGING_PATH = f'{prefix_path}/data/logs/log_attention_phd_qwen3_layer24.txt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

#### Load Dataset

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH,
)
print(f'Length of dataset: {len(dataset)}')

features = load_attention_dataset(FEATURES_PATH)
print(f'Length of features: {len(features)}')

In [ ]:
train_dataset, test_dataset = split_stratified(features, train_ratio=TRAIN_RATIO, random_state=42)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=32,
    collate_fn=attention_collate_fn,
    shuffle=True,
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=32,
    collate_fn=attention_collate_fn,
    shuffle=True,
)

#### Experiment

Two metrics evaluated independently:
- `vtar`: `1 - visual_token_attention_ratio` — low visual attention signals hallucination
- `sink`: attention-sink ratio on token-0 — high sink signals the model is ignoring visual content

In [ ]:
os.makedirs(os.path.dirname(LOGGING_PATH), exist_ok=True)
logging.basicConfig(filename=LOGGING_PATH, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

results = {}

for metric_type in ['vtar', 'sink']:
    detector = AttentionThresholdDetector(metric_type=metric_type)
    train_result = train_attention_detector(detector, train_dataloader)
    eval_result = eval_attention_detector(detector, test_dataloader)

    logging.info(f'[{metric_type}] threshold={train_result["threshold"]:.4f}, train_f1={train_result["f1"]:.4f}')
    logging.info(f'[{metric_type}] acc={eval_result["acc"]:.4f}, f1={eval_result["f1"]:.4f}, precision={eval_result["precision"]:.4f}, recall={eval_result["recall"]:.4f}, pr_auc={eval_result["pr_auc"]:.4f}')

    results[metric_type] = {
        'threshold': train_result['threshold'],
        'acc':       eval_result['acc'],
        'f1':        eval_result['f1'],
        'precision': eval_result['precision'],
        'recall':    eval_result['recall'],
        'pr_auc':    eval_result['pr_auc'],
    }

    print(f'[{metric_type}] threshold={train_result["threshold"]:.4f} | acc={eval_result["acc"]:.4f}, f1={eval_result["f1"]:.4f}, precision={eval_result["precision"]:.4f}, recall={eval_result["recall"]:.4f}, pr_auc={eval_result["pr_auc"]:.4f}')

logger = logging.getLogger()
for handler in logger.handlers[:]:
    handler.close()
    logger.removeHandler(handler)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)